# Regressão Linear — Previsão de Temperatura Máxima

**Objetivo:** Treinar e avaliar um modelo de Regressão Linear (Scikit-Learn) para prever a `temp_max`
a partir de variáveis climáticas históricas. Métricas: MAE, MSE, R².

In [4]:
pip install pandas==2.2.2 numpy==1.26.4 scikit-learn==1.5.0 matplotlib==3.9.0 seaborn==0.13.2 plotly==5.22.0 streamlit==1.35.0 tensorflow==2.16.1 torch==2.3.1 torchvision==0.18.1 jupyter==1.0.0 ipykernel==6.29.4 nbformat==5.10.4 scipy==1.13.1 joblib==1.4.2


Defaulting to user installation because normal site-packages is not writeable
  Using cached pandas-2.2.2.tar.gz (4.4 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'error'
Note: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  Preparing metadata (pyproject.toml) did not run successfully.
  exit code: 1
  
  [12 lines of output]
  + meson setup C:\Users\bahgo\AppData\Local\Temp\pip-install-esuv4u9p\pandas_ff4ad5744560414e96903b2a95fef0ce C:\Users\bahgo\AppData\Local\Temp\pip-install-esuv4u9p\pandas_ff4ad5744560414e96903b2a95fef0ce\.mesonpy-rm4bodnp\build -Dbuildtype=release -Db_ndebug=if-release -Db_vscrt=md --vsenv --native-file=C:\Users\bahgo\AppData\Local\Temp\pip-install-esuv4u9p\pandas_ff4ad5744560414e96903b2a95fef0ce\.mesonpy-rm4bodnp\build\meson-python-native-file.ini
  The Meson build system
  Version: 1.2.1
  Source dir: C:\Users\bahgo\AppData\Local\Temp\pip-install-esuv4u9p\pandas_ff4ad5744560414e96903b2a95fef0ce
  Build dir: C:\Users\bahgo\AppData\Local\Temp\pip-install-esuv4u9p\pandas_ff4ad5744560414e96903b2a95fef0ce\.mesonpy-rm4bodnp\build
  Build type: native build
  Project name: pandas
  Project version: 2.2.2
  
  ..\..\meson.build:2:0: ERROR: Could 

In [5]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
import joblib

from src.data.preprocessor import get_processed_data

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120
Path('../outputs/figures').mkdir(parents=True, exist_ok=True)
Path('../outputs/models').mkdir(parents=True, exist_ok=True)
print('OK')

ModuleNotFoundError: No module named 'sklearn'

## 1. Preparação dos Dados

In [ ]:
df = get_processed_data('../data/raw/cerrado_clima_raw.csv')

FEATURES = [
    'temp_min', 'umidade_relativa', 'velocidade_vento',
    'radiacao_solar', 'precipitacao', 'mes_sin', 'mes_cos',
    'dia_sin', 'dia_cos', 'amplitude_termica', 'precip_7d', 'temp_media_30d'
]
TARGET = 'temp_max'

df_model = df[FEATURES + [TARGET]].dropna()
X = df_model[FEATURES]
y = df_model[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print(f'Treino: {X_train.shape} | Teste: {X_test.shape}')

## 2. Treinamento — Regressão Linear Simples e Ridge

In [ ]:
models = {
    'Regressão Linear': LinearRegression(),
    'Ridge (α=1.0)': Ridge(alpha=1.0),
}

results = {}
for name, model in models.items():
    model.fit(X_train_s, y_train)
    y_pred = model.predict(X_test_s)
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    cv = cross_val_score(model, X_train_s, y_train, cv=5, scoring='r2').mean()
    results[name] = {'MAE': mae, 'MSE': mse, 'RMSE': rmse, 'R²': r2, 'R² CV-5': cv, 'y_pred': y_pred}
    print(f'{name}  →  MAE={mae:.3f}°C | RMSE={rmse:.3f}°C | R²={r2:.4f} | CV R²={cv:.4f}')

## 3. Visualização — Real vs. Predito

In [ ]:
best_name = max({k: v['R²'] for k, v in results.items()}, key=lambda k: results[k]['R²'])
y_pred_best = results[best_name]['y_pred']

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Scatter: real vs predito
ax = axes[0]
ax.scatter(y_test, y_pred_best, alpha=0.3, s=5, color='steelblue')
lim = [y_test.min() - 1, y_test.max() + 1]
ax.plot(lim, lim, 'r--', linewidth=2, label='Linha perfeita')
ax.set_xlabel('Temperatura Real (°C)')
ax.set_ylabel('Temperatura Predita (°C)')
ax.set_title(f'{best_name}\nR²={results[best_name]["R²"]:.4f} | MAE={results[best_name]["MAE"]:.3f}°C')
ax.legend()

# Série temporal: 90 dias
ax2 = axes[1]
idx = y_test.index[-90:]
ax2.plot(idx, y_test[-90:].values, label='Real', color='steelblue')
ax2.plot(idx, y_pred_best[-90:], label='Predito', color='tomato', linestyle='--')
ax2.set_xlabel('Data')
ax2.set_ylabel('Temperatura Máxima (°C)')
ax2.set_title('Últimos 90 dias do conjunto de teste')
ax2.legend()
plt.xticks(rotation=30)

plt.tight_layout()
plt.savefig('../outputs/figures/02_regressao_resultados.png')
plt.show()

## 4. Importância das Features (Coeficientes)

In [ ]:
lr = models['Regressão Linear']
coef_df = pd.DataFrame({'Feature': FEATURES, 'Coeficiente': lr.coef_})
coef_df = coef_df.reindex(coef_df['Coeficiente'].abs().sort_values(ascending=False).index)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['tomato' if c > 0 else 'steelblue' for c in coef_df['Coeficiente']]
ax.barh(coef_df['Feature'], coef_df['Coeficiente'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Coeficiente (features padronizadas)')
ax.set_title('Importância das Features — Regressão Linear')
plt.tight_layout()
plt.savefig('../outputs/figures/02_coeficientes.png')
plt.show()

## 5. Análise de Resíduos

In [ ]:
residuos = y_test.values - y_pred_best

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_pred_best, residuos, alpha=0.3, s=5, color='purple')
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set_xlabel('Predito')
axes[0].set_ylabel('Resíduo')
axes[0].set_title('Resíduos vs. Predito')

axes[1].hist(residuos, bins=60, color='purple', alpha=0.7, edgecolor='black')
axes[1].set_xlabel('Resíduo (°C)')
axes[1].set_ylabel('Frequência')
axes[1].set_title('Distribuição dos Resíduos')

plt.tight_layout()
plt.savefig('../outputs/figures/02_residuos.png')
plt.show()
print(f'Resíduo médio: {residuos.mean():.4f}°C | Desvio padrão: {residuos.std():.4f}°C')

In [ ]:
# Salva modelo e scaler
joblib.dump(models['Regressão Linear'], '../outputs/models/regressao_linear.pkl')
joblib.dump(scaler, '../outputs/models/scaler_regressao.pkl')
print('Modelo e scaler salvos em outputs/models/')

print('\n=== Tabela Final de Métricas ===')
summary = pd.DataFrame({k: {m: v for m, v in vals.items() if m != 'y_pred'} for k, vals in results.items()}).T
print(summary.round(4))